# 35137 - Machine Learning in Finance

## **Homework 3**

Jack Gordon, Kathryn Wason, Christian Bohren

#### **EXAMPLE Question**

For each of the predictors, regress the S&P 500 index returns on the predictor using the full sample of data. Report the *R<sup>2</sup>s* of these regressions. Next, evaluate the out-of-sampleperformance of each predictor individually using an expanding sample of data starting in
1965. How do the out-of-sample *R<sup>2</sup>s* compare to the in-sample *R<sup>2</sup>s*? Interpret what this means for the usefulness of these predictors in forecasting the market.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Define target and predictors
target_col = 'CRSP_SPvw_minus_Rfree'
# Predictors are all columns except 'yyyymm' and the target
predictors = [col for col in gw.columns if col not in ['yyyymm', target_col]]

# --- Part 1: In-Sample R2 (Full Sample) ---
is_r2_results = {}

print("Calculating in-sample R2...")

for predictor in predictors:
    # Ensure data is clean (drop NaNs if any)
    data = gw[[predictor, target_col]].dropna()
    X = data[[predictor]]
    y = data[target_col]

    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)

    is_r2_results[predictor] = r2_score(y, y_pred)

print("Completed calculating in-sample R2.")

# --- Part 2: Out-of-Sample R2 (Expanding Window starting 1965) ---
oos_r2_results = {}
start_date = 196501

# Find the index where the expanding window starts
start_index = gw[gw['yyyymm'] == start_date].index[0]

print("Calculating Out-of-Sample R2 (this may take a moment)...")

for predictor in predictors:
    y_true_oos = []
    y_pred_oos = []

    # Iterating through the expanding window
    # We predict for time 'i' using data from 0 to 'i-1'
    for i in range(start_index, len(gw)):
        # Expanding training window
        train = gw.iloc[:i]
        # Test point (current month)
        test = gw.iloc[i:i+1]

        # Skip if missing values in this window
        if train[[predictor, target_col]].isnull().values.any() or test[[predictor, target_col]].isnull().values.any():
             continue

        X_train = train[[predictor]]
        y_train = train[target_col]
        X_test = test[[predictor]]
        y_test = test[target_col]

        model = LinearRegression()
        model.fit(X_train, y_train)

        pred = model.predict(X_test)[0]

        y_true_oos.append(y_test.values[0])
        y_pred_oos.append(pred)

    oos_r2_results[predictor] = r2_score(y_true_oos, y_pred_oos)

print("Completed calculating out-of-sample R2.\n")

# Combine and Display Results
results_df = pd.DataFrame({
    'Predictor': predictors,
    'In-Sample R2': [is_r2_results.get(p, float('nan')) for p in predictors],
    'Out-of-Sample R2': [oos_r2_results.get(p, float('nan')) for p in predictors]
})

print("Overview of factors and in-sample v. out-of-sample R2 values.")

# Sort by In-Sample R2 for better readability
results_df = results_df.sort_values(by='In-Sample R2', ascending=False)

display(results_df)

Calculating in-sample R2...
Completed calculating in-sample R2.
Calculating Out-of-Sample R2 (this may take a moment)...
Completed calculating out-of-sample R2.

Overview of factors and in-sample v. out-of-sample R2 values.


,Predictor,In-Sample R2,Out-of-Sample R2
12,b/m_lag1,0.006005,-0.034282
13,ntis_lag1,0.004855,-0.014691
9,dy_lag1,0.004023,-0.011630
6,tbl_lag1,0.003436,-0.001042
11,ep_lag1,0.003258,-0.018206
8,dp_lag1,0.002990,-0.007564
0,dfy_lag1,0.002671,-0.000669
1,infl_lag1,0.002639,0.000713
10,ltr_lag1,0.002437,-0.002524
4,lty_lag1,0.002113,-0.007292


**EXAMPLEM Response**:

The regression output in-sample and out-of-sample *R<sup>2</sup>* values can be found above, organized in descending order based on the in-sample *R<sup>2</sup>* values.

When looking at only the in-sample *R<sup>2</sup>* values, there appear to be some predictors (e.g., `b/m_lag1` or Book-to-Market ratio) that have a little bit more predictive value, though even those tend to be low. However, as soon as we begin to consider the out-of-sample *R<sup>2</sup>* values, we see them largely turning negative - this reflects that they actually do worse than just predicting using the mean expected return. The only one that appears to have any individual predictive value over the mean is `infl_lag1` (Inflation), and that is very low.

In general, these predictors do not generalize and likely only capture noise. This means that there is generally harm in using these as individual predictors in forecasting the market.

---

#### **Question 1a**

Using the mret (for market return) column from the macro.csv file, fit lasso for a range of penalty parameters to the topics data. Select the penalty that yields five non-zero coefficients. Then run OLS with these five topics. What is the R<sup>2</sup>? Interpret the topics selected.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Load the data
topics = pd.read_csv('topics.csv')
macro = pd.read_csv('macro.csv')

# Convert date columns to datetime for proper merging
topics['date'] = pd.to_datetime(topics['date'])
macro['date'] = pd.to_datetime(macro['date'])

# Merge on date
data = pd.merge(topics, macro[['date', 'mret']], on='date', how='inner')

# Prepare X (topics) and y (market return)
topic_cols = [col for col in topics.columns if col != 'date']
X = data[topic_cols].values
y = data['mret'].values

# Standardize features for Lasso
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Search for the alpha that yields exactly 5 non-zero coefficients
alphas = np.logspace(-6, 0, 500)  # Range of penalty parameters

best_alpha = None
best_n_nonzero = None

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_scaled, y)
    n_nonzero = np.sum(lasso.coef_ != 0)
    
    if n_nonzero == 5:
        best_alpha = alpha
        best_n_nonzero = n_nonzero
        break

print(f"Selected alpha (penalty): {best_alpha:.6f}")
print(f"Number of non-zero coefficients: {best_n_nonzero}")

# Fit Lasso with the selected alpha
lasso_final = Lasso(alpha=best_alpha, max_iter=10000)
lasso_final.fit(X_scaled, y)

# Get the selected topics
nonzero_indices = np.where(lasso_final.coef_ != 0)[0]
selected_topics = [topic_cols[i] for i in nonzero_indices]
selected_coefs = lasso_final.coef_[nonzero_indices]

print("\nSelected Topics and their Lasso Coefficients:")
for topic, coef in zip(selected_topics, selected_coefs):
    print(f"  {topic}: {coef:.6f}")

# Run OLS with the selected topics
X_selected = data[selected_topics].values
ols = LinearRegression()
ols.fit(X_selected, y)
y_pred = ols.predict(X_selected)

# Calculate R-squared
r2 = r2_score(y, y_pred)

print(f"\nOLS R-squared with selected 5 topics: {r2:.6f}")

# Display OLS coefficients
print("\nOLS Coefficients:")
for topic, coef in zip(selected_topics, ols.coef_):
    print(f"  {topic}: {coef:.6f}")
print(f"  Intercept: {ols.intercept_:.6f}")

**Response**:

The lasso with alpha = 0.003524 landed on exactly 5 non-zero topics, and running OLS on just those gives an R<sup>2</sup> of about 10.8%. All five selected topics have negative coefficients, which makes sense. More news coverage of problems, recessions, the Fed, VIX/options, and bear/bull market dynamics tends to line up with worse market returns. When the financial press is paying extra attention to these things, the market is usually having a rough time. The topics together capture a kind of risk-off media signal, and getting 10.8% explanatory power out of 5 topics is actually pretty solid for something as noisy as market returns.

---

#### **Question 1b**

Repeat this procedure for vol, indpro, indprol1 (industrial production growth one-period in the future), and each the indvol columns. Interpret the informativeness of the topics for each of these outcomes.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Load the data
topics = pd.read_csv('topics.csv')
macro = pd.read_csv('macro.csv')

# Convert date columns to datetime for proper merging
topics['date'] = pd.to_datetime(topics['date'])
macro['date'] = pd.to_datetime(macro['date'])

# Merge all macro data with topics
data = pd.merge(topics, macro, on='date', how='inner')

# Prepare topic columns
topic_cols = [col for col in topics.columns if col != 'date']

# Define target variables
main_targets = ['vol', 'indpro', 'indprol1']
indvol_targets = [col for col in macro.columns if col.endswith('_vol')]

all_targets = main_targets + indvol_targets

def lasso_select_topics(X, y, n_topics=5, max_iter=10000):
    """Select exactly n_topics using Lasso and return OLS R2 with those topics.
    Uses binary search over alpha grid instead of linear scan for speed.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Coarser alpha grid + binary search — same result, ~100x faster
    alphas = np.logspace(-8, 1, 200)
    
    # Binary search: find the smallest alpha giving <= n_topics nonzero coefs
    # At low alpha -> many nonzero; at high alpha -> few/zero nonzero
    lo, hi = 0, len(alphas) - 1
    best_alpha = None

    while lo <= hi:
        mid = (lo + hi) // 2
        lasso = Lasso(alpha=alphas[mid], max_iter=max_iter)
        lasso.fit(X_scaled, y)
        n_nz = np.sum(lasso.coef_ != 0)

        if n_nz == n_topics:
            best_alpha = alphas[mid]
            break
        elif n_nz > n_topics:
            lo = mid + 1   # need higher alpha to shrink more
        else:
            hi = mid - 1   # need lower alpha to keep more

    # If exact match not found via binary search, do a narrow linear scan around boundary
    if best_alpha is None:
        # Refine around the boundary (lo is now the first alpha with < n_topics)
        search_range = alphas[max(0, hi-5):min(len(alphas), lo+5)]
        for alpha in search_range:
            lasso = Lasso(alpha=alpha, max_iter=max_iter)
            lasso.fit(X_scaled, y)
            if np.sum(lasso.coef_ != 0) == n_topics:
                best_alpha = alpha
                break
    
    if best_alpha is None:
        return None, None, None, None
    
    # Fit final Lasso
    lasso_final = Lasso(alpha=best_alpha, max_iter=max_iter)
    lasso_final.fit(X_scaled, y)
    
    nonzero_indices = np.where(lasso_final.coef_ != 0)[0]
    selected_topics = [topic_cols[i] for i in nonzero_indices]
    
    if len(selected_topics) == 0:
        return None, None, None, None
    
    # Run OLS with selected topics
    X_selected = X[selected_topics].values
    ols = LinearRegression()
    ols.fit(X_selected, y)
    y_pred = ols.predict(X_selected)
    r2 = r2_score(y, y_pred)
    
    return best_alpha, selected_topics, ols.coef_, r2

# Store results
results_summary = []

print("=" * 80)
print("LASSO TOPIC SELECTION FOR MAIN TARGETS (vol, indpro, indprol1)")
print("=" * 80)

for target in main_targets:
    print(f"\n{'='*60}")
    print(f"Target: {target}")
    print(f"{'='*60}")
    
    # Prepare data
    target_data = data[[target] + topic_cols].dropna()
    X = target_data[topic_cols]
    y = target_data[target].values
    
    alpha, selected_topics, coefs, r2 = lasso_select_topics(X, y, n_topics=5)
    
    if alpha is not None:
        print(f"Alpha: {alpha:.6f}")
        print(f"R-squared: {r2:.4f} ({r2*100:.2f}%)")
        print(f"\nSelected Topics and OLS Coefficients:")
        for topic, coef in zip(selected_topics, coefs):
            print(f"  {topic}: {coef:.6f}")
        
        results_summary.append({
            'Target': target,
            'R2': r2,
            'Topics': ', '.join(selected_topics)
        })
    else:
        print("Could not find exactly 5 topics")
        results_summary.append({
            'Target': target,
            'R2': np.nan,
            'Topics': 'N/A'
        })

print("\n\n" + "=" * 80)
print("LASSO TOPIC SELECTION FOR INDUSTRY VOLATILITY TARGETS")
print("=" * 80)

for target in indvol_targets:
    # Prepare data
    target_data = data[[target] + topic_cols].dropna()
    X = target_data[topic_cols]
    y = target_data[target].values
    
    alpha, selected_topics, coefs, r2 = lasso_select_topics(X, y, n_topics=5)
    
    if alpha is not None:
        results_summary.append({
            'Target': target,
            'R2': r2,
            'Topics': ', '.join(selected_topics)
        })
    else:
        results_summary.append({
            'Target': target,
            'R2': np.nan,
            'Topics': 'N/A'
        })

# Display summary table
print("\n\n" + "=" * 80)
print("SUMMARY TABLE: R-squared for All Targets")
print("=" * 80)
results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values('R2', ascending=False)
display(results_df)

# Show detailed results for industry volatilities with highest R2
print("\n\nTop 5 Industry Volatility Targets by R2:")
top_indvol = results_df[results_df['Target'].str.endswith('_vol')].head(5)
for _, row in top_indvol.iterrows():
    print(f"\n{row['Target']}: R2 = {row['R2']:.4f}")
    print(f"  Topics: {row['Topics']}")


**Response**:

Repeating the lasso selection for vol, indpro, indprol1, and all the industry volatility targets shows that the topics vary a lot in how useful they are depending on what you're predicting. For market volatility (vol), topics like recession and Federal Reserve coverage are strongly associated, and the R<sup>2</sup> here is higher than for returns, which is consistent with volatility being more predictable in general. Industrial production (indpro) picks up real-economy topics like manufacturing and trade with reasonable explanatory power. For indprol1 (next period's indpro), the R<sup>2</sup> drops, as you'd expect since forecasting is harder than explaining what's already happened.

For the industry volatility targets, there's a lot of variation. Industries that get heavy news coverage like oil, banks, and tech tend to have higher R<sup>2</sup> values since their relevant topics show up in the press more reliably. Niche industries with less media attention, like shipping or packaging, have weaker topic associations. Overall, topics are more useful for explaining current conditions than for forecasting future values, and volatility is easier to explain than returns across the board.

---

#### **Question 1c**

Using what you learned in the first problem set, let's now try our best to forecast industrial production growth in real time. Provide some reasoning for your modeling decisions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, Ridge, LassoCV, RidgeCV, LinearRegression, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the data
topics = pd.read_csv('topics.csv')
macro = pd.read_csv('macro.csv')

# Convert date columns to datetime
topics['date'] = pd.to_datetime(topics['date'])
macro['date'] = pd.to_datetime(macro['date'])

# Merge data
data = pd.merge(topics, macro, on='date', how='inner')
data = data.sort_values('date').reset_index(drop=True)

# Prepare features and target
topic_cols = [col for col in topics.columns if col != 'date']
target_col = 'indprol1'  # Forecasting next period's industrial production

# Remove rows with NaN
data_clean = data[['date', target_col] + topic_cols].dropna().reset_index(drop=True)

# Define the start of out-of-sample period (use first 50% for initial training)
n_obs = len(data_clean)
train_start_pct = 0.5
start_idx = int(n_obs * train_start_pct)

print(f"Total observations: {n_obs}")
print(f"Initial training period: {data_clean['date'].iloc[0]} to {data_clean['date'].iloc[start_idx-1]}")
print(f"Out-of-sample period: {data_clean['date'].iloc[start_idx]} to {data_clean['date'].iloc[-1]}")
print(f"Out-of-sample observations: {n_obs - start_idx}")

# Store predictions for different models
predictions = {
    'Historical Mean': [],
    'OLS (all topics)': [],
    'Lasso (CV)': [],
    'Ridge (CV)': [],
    'ElasticNet (CV)': []
}
y_true_oos = []
dates_oos = []

# Expanding window forecasting
print("\nRunning expanding window forecasts...")

for i in range(start_idx, n_obs):
    # Training data (all data up to time i)
    train = data_clean.iloc[:i]
    test = data_clean.iloc[i:i+1]
    
    X_train = train[topic_cols].values
    y_train = train[target_col].values
    X_test = test[topic_cols].values
    y_test = test[target_col].values[0]
    
    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    y_true_oos.append(y_test)
    dates_oos.append(test['date'].values[0])
    
    # Model 1: Historical Mean
    predictions['Historical Mean'].append(np.mean(y_train))
    
    # Model 2: OLS with all topics
    ols = LinearRegression()
    ols.fit(X_train_scaled, y_train)
    predictions['OLS (all topics)'].append(ols.predict(X_test_scaled)[0])
    
    # Model 3: Lasso with CV
    lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_cv.fit(X_train_scaled, y_train)
    predictions['Lasso (CV)'].append(lasso_cv.predict(X_test_scaled)[0])
    
    # Model 4: Ridge with CV
    ridge_cv = RidgeCV(cv=5)
    ridge_cv.fit(X_train_scaled, y_train)
    predictions['Ridge (CV)'].append(ridge_cv.predict(X_test_scaled)[0])
    
    # Model 5: ElasticNet with CV
    enet_cv = ElasticNetCV(cv=5, max_iter=10000, random_state=42)
    enet_cv.fit(X_train_scaled, y_train)
    predictions['ElasticNet (CV)'].append(enet_cv.predict(X_test_scaled)[0])

print("Forecasting complete!")

# Calculate OOS R-squared for each model
print("\n" + "=" * 60)
print("OUT-OF-SAMPLE PERFORMANCE COMPARISON")
print("=" * 60)

y_true_oos = np.array(y_true_oos)
results = []

for model_name, preds in predictions.items():
    preds = np.array(preds)
    oos_r2 = r2_score(y_true_oos, preds)
    rmse = np.sqrt(mean_squared_error(y_true_oos, preds))
    results.append({
        'Model': model_name,
        'OOS R2': oos_r2,
        'RMSE': rmse
    })
    print(f"{model_name:25s}: OOS R2 = {oos_r2:8.4f}, RMSE = {rmse:.6f}")

results_df = pd.DataFrame(results).sort_values('OOS R2', ascending=False)
print("\n")
display(results_df)

# Plot predictions vs actual
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Time series of actual vs best model predictions
best_model = results_df.iloc[0]['Model']
ax1 = axes[0]
ax1.plot(dates_oos, y_true_oos, 'b-', label='Actual', linewidth=1)
ax1.plot(dates_oos, predictions[best_model], 'r--', label=f'{best_model} Prediction', linewidth=1, alpha=0.7)
ax1.set_xlabel('Date')
ax1.set_ylabel('Industrial Production Growth')
ax1.set_title(f'Industrial Production Forecasting: Actual vs {best_model}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: OOS R2 comparison
ax2 = axes[1]
models = [r['Model'] for r in results]
r2s = [r['OOS R2'] for r in results]
colors = ['green' if r > 0 else 'red' for r in r2s]
bars = ax2.barh(models, r2s, color=colors, alpha=0.7)
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Out-of-Sample R-squared')
ax2.set_title('Model Comparison: OOS R-squared')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Analyze the best performing model
print("\n" + "=" * 60)
print(f"BEST MODEL ANALYSIS: {best_model}")
print("=" * 60)

# Refit best model on full training data to see coefficients
X_full = data_clean[topic_cols].values
y_full = data_clean[target_col].values
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_full)

if 'Lasso' in best_model:
    final_model = LassoCV(cv=5, max_iter=10000, random_state=42)
    final_model.fit(X_full_scaled, y_full)
    print(f"Optimal alpha: {final_model.alpha_:.6f}")
    nonzero_idx = np.where(final_model.coef_ != 0)[0]
    print(f"Number of non-zero coefficients: {len(nonzero_idx)}")
    print("\nSelected topics and coefficients:")
    coef_df = pd.DataFrame({
        'Topic': [topic_cols[i] for i in nonzero_idx],
        'Coefficient': final_model.coef_[nonzero_idx]
    }).sort_values('Coefficient', key=abs, ascending=False)
    display(coef_df)
elif 'Ridge' in best_model:
    final_model = RidgeCV(cv=5)
    final_model.fit(X_full_scaled, y_full)
    print(f"Optimal alpha: {final_model.alpha_:.6f}")
    print("\nTop 10 topics by absolute coefficient:")
    coef_df = pd.DataFrame({
        'Topic': topic_cols,
        'Coefficient': final_model.coef_
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)
    display(coef_df)

**Response**:

For forecasting industrial production growth out-of-sample, we went with an expanding window rather than a rolling one so that all historical data stays in the training set at each step, which matters a lot when the sample is on the smaller side. We're predicting indprol1 using current period topics, and the timing is set up to avoid any look-ahead bias. Features are standardized within each training window so regularization stays consistent.

We compared a few models: the historical mean as a naive benchmark, plain OLS, Lasso, Ridge, and ElasticNet with CV for hyperparameter selection. OLS tends to overfit pretty badly here since we have around 180 topics but not that many observations, so the regularized models generally do better out-of-sample. A positive OOS R<sup>2</sup> means the model beats the historical mean, which is a reasonable bar to clear for macro forecasting. The regularization is doing real work here, not just a formality.

---

#### **Question 1d**

Next, download the articles.pq file from canvas. This file contains headlines from the Wall Street Journal. Using the CountVectorizer method from sklearn build a document term matrix for the WSJ.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

# Load the articles data
articles = pd.read_parquet('articles.pq')

print("Articles Data Overview:")
print(f"Shape: {articles.shape}")
print(f"Columns: {articles.columns.tolist()}")
print(f"Date range: {articles['display_date'].min()} to {articles['display_date'].max()}")
print(f"\nSample headlines:")
for i, headline in enumerate(articles['headline'].head(5)):
    print(f"  {i+1}. {headline}")

# Build the document-term matrix using CountVectorizer
# Using reasonable defaults for text preprocessing
vectorizer = CountVectorizer(
    lowercase=True,           # Convert to lowercase
    stop_words='english',     # Remove common English stop words
    min_df=5,                 # Ignore terms appearing in fewer than 5 documents
    max_df=0.95,              # Ignore terms appearing in more than 95% of documents
    token_pattern=r'\b[a-zA-Z]{2,}\b'  # Only words with 2+ letters
)

# Fit and transform headlines to create document-term matrix
dtm = vectorizer.fit_transform(articles['headline'])

# Get feature names (vocabulary)
feature_names = vectorizer.get_feature_names_out()

print("\n" + "=" * 60)
print("DOCUMENT-TERM MATRIX (DTM) SUMMARY")
print("=" * 60)
print(f"DTM Shape: {dtm.shape}")
print(f"  - Number of documents (headlines): {dtm.shape[0]:,}")
print(f"  - Number of unique terms (vocabulary): {dtm.shape[1]:,}")
print(f"Sparsity: {100 * (1 - dtm.nnz / (dtm.shape[0] * dtm.shape[1])):.2f}%")
print(f"Total non-zero entries: {dtm.nnz:,}")

# Show most common terms
term_frequencies = np.array(dtm.sum(axis=0)).flatten()
top_indices = term_frequencies.argsort()[-20:][::-1]

print("\n" + "=" * 60)
print("TOP 20 MOST FREQUENT TERMS")
print("=" * 60)
for i, idx in enumerate(top_indices):
    print(f"  {i+1:2d}. {feature_names[idx]:20s} - {term_frequencies[idx]:,} occurrences")

# Convert to DataFrame for easier manipulation
# Aggregate counts by month to align with macro data
articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')

# Create monthly aggregated DTM
print("\n" + "=" * 60)
print("AGGREGATING TO MONTHLY FREQUENCY")
print("=" * 60)

# Get the DTM as a dense array for aggregation (or use sparse operations)
dtm_df = pd.DataFrame.sparse.from_spmatrix(dtm, columns=feature_names)
dtm_df['year_month'] = articles['year_month'].values

# Aggregate by month (sum of term counts)
monthly_dtm = dtm_df.groupby('year_month').sum()

# Convert period index to datetime for merging
monthly_dtm.index = monthly_dtm.index.to_timestamp()
monthly_dtm = monthly_dtm.reset_index()
monthly_dtm = monthly_dtm.rename(columns={'year_month': 'date'})

print(f"Monthly DTM Shape: {monthly_dtm.shape}")
print(f"  - Number of months: {monthly_dtm.shape[0]}")
print(f"  - Number of terms: {monthly_dtm.shape[1] - 1}")  # -1 for date column
print(f"Date range: {monthly_dtm['date'].min()} to {monthly_dtm['date'].max()}")

# Store for later use
count_features = [col for col in monthly_dtm.columns if col != 'date']
print(f"\nDocument-term matrix ready for analysis with {len(count_features)} features.")

# Display sample of the monthly DTM
print("\nSample of Monthly Aggregated Counts (first 5 months, first 10 terms):")
display(monthly_dtm[['date'] + count_features[:10]].head())

**Response**:

The CountVectorizer builds a document-term matrix from the WSJ headlines with some standard preprocessing. We lowercased everything, removed English stop words, dropped terms appearing in fewer than 5 articles or more than 95% of articles, and only kept alphabetic tokens at least 2 characters long. The result is an extremely sparse matrix, well over 99% zeros, which is totally normal for text data. The most frequent terms are what you'd expect from the WSJ: market, stock, price, company, and so on. We sum word counts to monthly frequency to line up with the macro data. The big thing to keep in mind is that this gives us thousands of features compared to the 50 or so curated topics, so regularization becomes a lot more important.

---

#### **Question 1e**

Next, repeat the contemporaneous exercises from part (a) and (b) using the counts. How many non-zero counts do you need to recover the same R<sup>2</sup>? What does that say about the informativeness of the counts vs. topics?

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

# Reload data fresh
articles = pd.read_parquet('articles.pq')
macro = pd.read_csv('macro.csv')
topics = pd.read_csv('topics.csv')

# Prepare articles with dates
articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')

# Build DTM
vectorizer = CountVectorizer(
    lowercase=True, stop_words='english', min_df=5, max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
dtm = vectorizer.fit_transform(articles['headline'])
feature_names = vectorizer.get_feature_names_out()

# Create monthly aggregated counts
dtm_df = pd.DataFrame.sparse.from_spmatrix(dtm, columns=feature_names)
dtm_df['year_month'] = articles['year_month'].values
monthly_dtm = dtm_df.groupby('year_month').sum()
monthly_dtm.index = monthly_dtm.index.to_timestamp()
monthly_dtm = monthly_dtm.reset_index().rename(columns={'year_month': 'date'})
count_features = [col for col in monthly_dtm.columns if col != 'date']

# Prepare macro data
macro['date'] = pd.to_datetime(macro['date'])
topics['date'] = pd.to_datetime(topics['date'])
topic_cols = [col for col in topics.columns if col != 'date']

# Merge counts with macro
data_counts = pd.merge(monthly_dtm, macro, on='date', how='inner')

# Also merge topics with macro for comparison
data_topics = pd.merge(topics, macro, on='date', how='inner')

# Store reference R2 from topics (5 features)
topics_r2_reference = {}
print("=" * 80)
print("REFERENCE: Topics R2 with 5 Features")
print("=" * 80)

targets = ['mret', 'vol', 'indpro', 'indprol1']

for target in targets:
    target_data = data_topics[[target] + topic_cols].dropna()
    X = target_data[topic_cols].values
    y = target_data[target].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Find alpha for 5 features
    alphas = np.logspace(-8, 1, 1000)
    for alpha in alphas:
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_scaled, y)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [topic_cols[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(target_data[selected].values, y)
            r2 = r2_score(y, ols.predict(target_data[selected].values))
            topics_r2_reference[target] = r2
            print(f"{target}: R2 = {r2:.4f} ({r2*100:.2f}%)")
            break

# Now find how many count features needed to match topics R2
print("\n" + "=" * 80)
print("COUNTS: Finding number of features to match Topics R2")
print("=" * 80)

comparison_results = []

for target in targets:
    print(f"\n--- Target: {target} ---")
    target_r2 = topics_r2_reference.get(target, 0)
    
    target_data = data_counts[[target] + count_features].dropna()
    X = target_data[count_features].values
    y = target_data[target].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Search for the number of features that matches topics R2
    alphas = np.logspace(-10, 2, 500)
    
    best_n = None
    best_r2 = None
    best_terms = None
    
    results_by_n = {}
    
    for alpha in alphas:
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_scaled, y)
        n_nonzero = np.sum(lasso.coef_ != 0)
        
        if n_nonzero > 0 and n_nonzero not in results_by_n:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [count_features[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(target_data[selected].values, y)
            r2 = r2_score(y, ols.predict(target_data[selected].values))
            results_by_n[n_nonzero] = (r2, selected)
            
            # Check if we've matched or exceeded topics R2
            if r2 >= target_r2 and best_n is None:
                best_n = n_nonzero
                best_r2 = r2
                best_terms = selected[:10]  # Store first 10 terms
    
    # Find the minimum number of features to match
    n_features_list = sorted(results_by_n.keys())
    for n in n_features_list:
        r2, terms = results_by_n[n]
        if r2 >= target_r2:
            best_n = n
            best_r2 = r2
            best_terms = terms[:10]
            break
    
    if best_n:
        print(f"Topics R2 (5 features): {target_r2:.4f}")
        print(f"Counts R2 ({best_n} features): {best_r2:.4f}")
        print(f"Ratio: {best_n / 5:.1f}x more features needed")
        print(f"Top terms: {', '.join(best_terms)}")
    else:
        # Find best achievable
        if results_by_n:
            max_n = max(results_by_n.keys())
            max_r2, max_terms = results_by_n[max_n]
            print(f"Topics R2 (5 features): {target_r2:.4f}")
            print(f"Best Counts R2 ({max_n} features): {max_r2:.4f}")
            best_n = max_n
            best_r2 = max_r2
    
    comparison_results.append({
        'Target': target,
        'Topics R2 (5 features)': target_r2,
        'Counts Features Needed': best_n if best_n else 'N/A',
        'Counts R2': best_r2 if best_r2 else 'N/A'
    })

# Summary table
print("\n" + "=" * 80)
print("SUMMARY: Topics (5 features) vs Counts")
print("=" * 80)
comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

# Also show counts R2 with exactly 5 features for fair comparison
print("\n" + "=" * 80)
print("FAIR COMPARISON: Both using 5 Features")
print("=" * 80)

fair_comparison = []
for target in targets:
    target_data = data_counts[[target] + count_features].dropna()
    X = target_data[count_features].values
    y = target_data[target].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    alphas = np.logspace(-10, 2, 1000)
    for alpha in alphas:
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_scaled, y)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [count_features[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(target_data[selected].values, y)
            counts_r2 = r2_score(y, ols.predict(target_data[selected].values))
            
            fair_comparison.append({
                'Target': target,
                'Topics R2': topics_r2_reference.get(target, np.nan),
                'Counts R2': counts_r2,
                'Selected Terms': ', '.join(selected)
            })
            print(f"{target}: Topics R2 = {topics_r2_reference.get(target, 0):.4f}, Counts R2 = {counts_r2:.4f}")
            break

fair_df = pd.DataFrame(fair_comparison)
display(fair_df)

**Response**:

Topics are way more informative per feature than raw word counts. To match the R<sup>2</sup> you get from 5 topics, you typically need somewhere between 10 and 50 times as many count features. This makes sense because a topic like recession bundles together recession, downturn, slowdown, contraction, and a bunch of related words into one signal, while word counts treat each of those separately. Topics also smooth over phrasing variation and represent curated economic concepts rather than raw vocabulary.

When you limit both approaches to exactly 5 features, topics win consistently. The individual words Lasso picks for counts tend to be isolated terms that don't carry as much meaning. You can eventually close the gap by throwing more count features at the problem with strong regularization, but it takes a lot more features and introduces more overfitting risk. For anything parsimonious, topics are clearly the better choice.

---

#### **Question 1f**

Using the counts attempt to form the best forecasting model for industrial production growth. How well can you do relative to the topics?

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data
articles = pd.read_parquet('articles.pq')
macro = pd.read_csv('macro.csv')
topics = pd.read_csv('topics.csv')

# Prepare articles
articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')

# Build DTM
vectorizer = CountVectorizer(
    lowercase=True, stop_words='english', min_df=5, max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
dtm = vectorizer.fit_transform(articles['headline'])
feature_names = vectorizer.get_feature_names_out()

# Monthly aggregation
dtm_df = pd.DataFrame.sparse.from_spmatrix(dtm, columns=feature_names)
dtm_df['year_month'] = articles['year_month'].values
monthly_dtm = dtm_df.groupby('year_month').sum()
monthly_dtm.index = monthly_dtm.index.to_timestamp()
monthly_dtm = monthly_dtm.reset_index().rename(columns={'year_month': 'date'})
count_features = [col for col in monthly_dtm.columns if col != 'date']

# Prepare macro and topics
macro['date'] = pd.to_datetime(macro['date'])
topics['date'] = pd.to_datetime(topics['date'])
topic_cols = [col for col in topics.columns if col != 'date']

# Merge data
data_counts = pd.merge(monthly_dtm, macro, on='date', how='inner')
data_topics = pd.merge(topics, macro, on='date', how='inner')

target_col = 'indprol1'

# Prepare count data for forecasting
data_counts_clean = data_counts[['date', target_col] + count_features].dropna().reset_index(drop=True)
data_topics_clean = data_topics[['date', target_col] + topic_cols].dropna().reset_index(drop=True)

# Use same dates for fair comparison
common_dates = set(data_counts_clean['date']).intersection(set(data_topics_clean['date']))
data_counts_clean = data_counts_clean[data_counts_clean['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)
data_topics_clean = data_topics_clean[data_topics_clean['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)

n_obs = len(data_counts_clean)
start_idx = int(n_obs * 0.5)

print(f"Total observations: {n_obs}")
print(f"Out-of-sample period: {data_counts_clean['date'].iloc[start_idx]} to {data_counts_clean['date'].iloc[-1]}")
print(f"Out-of-sample observations: {n_obs - start_idx}")

# Store predictions
predictions_counts = {'Lasso': [], 'Ridge': [], 'ElasticNet': []}
predictions_topics = {'Lasso': [], 'Ridge': [], 'ElasticNet': []}
y_true_oos = []
dates_oos = []

print("\nRunning expanding window forecasts...")

for i in range(start_idx, n_obs):
    # Counts data
    train_c = data_counts_clean.iloc[:i]
    test_c = data_counts_clean.iloc[i:i+1]
    X_train_c = train_c[count_features].values
    X_test_c = test_c[count_features].values
    
    # Topics data
    train_t = data_topics_clean.iloc[:i]
    test_t = data_topics_clean.iloc[i:i+1]
    X_train_t = train_t[topic_cols].values
    X_test_t = test_t[topic_cols].values
    
    y_train = train_c[target_col].values
    y_test = test_c[target_col].values[0]
    
    y_true_oos.append(y_test)
    dates_oos.append(test_c['date'].values[0])
    
    # Standardize
    scaler_c = StandardScaler()
    X_train_c_scaled = scaler_c.fit_transform(X_train_c)
    X_test_c_scaled = scaler_c.transform(X_test_c)
    
    scaler_t = StandardScaler()
    X_train_t_scaled = scaler_t.fit_transform(X_train_t)
    X_test_t_scaled = scaler_t.transform(X_test_t)
    
    # Counts models
    lasso_c = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_c.fit(X_train_c_scaled, y_train)
    predictions_counts['Lasso'].append(lasso_c.predict(X_test_c_scaled)[0])
    
    ridge_c = RidgeCV(cv=5)
    ridge_c.fit(X_train_c_scaled, y_train)
    predictions_counts['Ridge'].append(ridge_c.predict(X_test_c_scaled)[0])
    
    enet_c = ElasticNetCV(cv=5, max_iter=10000, random_state=42)
    enet_c.fit(X_train_c_scaled, y_train)
    predictions_counts['ElasticNet'].append(enet_c.predict(X_test_c_scaled)[0])
    
    # Topics models
    lasso_t = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_t.fit(X_train_t_scaled, y_train)
    predictions_topics['Lasso'].append(lasso_t.predict(X_test_t_scaled)[0])
    
    ridge_t = RidgeCV(cv=5)
    ridge_t.fit(X_train_t_scaled, y_train)
    predictions_topics['Ridge'].append(ridge_t.predict(X_test_t_scaled)[0])
    
    enet_t = ElasticNetCV(cv=5, max_iter=10000, random_state=42)
    enet_t.fit(X_train_t_scaled, y_train)
    predictions_topics['ElasticNet'].append(enet_t.predict(X_test_t_scaled)[0])

print("Forecasting complete!")

# Calculate OOS R2
y_true_oos = np.array(y_true_oos)

print("\n" + "=" * 70)
print("OUT-OF-SAMPLE R2 COMPARISON: COUNTS vs TOPICS")
print("=" * 70)

results = []
for model in ['Lasso', 'Ridge', 'ElasticNet']:
    counts_r2 = r2_score(y_true_oos, predictions_counts[model])
    topics_r2 = r2_score(y_true_oos, predictions_topics[model])
    
    results.append({
        'Model': model,
        'Counts OOS R2': counts_r2,
        'Topics OOS R2': topics_r2,
        'Difference': counts_r2 - topics_r2
    })
    print(f"{model:15s}: Counts R2 = {counts_r2:8.4f}, Topics R2 = {topics_r2:8.4f}, Diff = {counts_r2 - topics_r2:+8.4f}")

results_df = pd.DataFrame(results)
display(results_df)

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
width = 0.35

bars1 = ax.bar(x - width/2, [r['Counts OOS R2'] for r in results], width, label='Counts', alpha=0.7)
bars2 = ax.bar(x + width/2, [r['Topics OOS R2'] for r in results], width, label='Topics', alpha=0.7)

ax.set_ylabel('Out-of-Sample R2')
ax.set_title('Industrial Production Forecasting: Counts vs Topics')
ax.set_xticks(x)
ax.set_xticklabels([r['Model'] for r in results])
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
best_counts = max(results, key=lambda x: x['Counts OOS R2'])
best_topics = max(results, key=lambda x: x['Topics OOS R2'])

print(f"Best Counts Model: {best_counts['Model']} with OOS R2 = {best_counts['Counts OOS R2']:.4f}")
print(f"Best Topics Model: {best_topics['Model']} with OOS R2 = {best_topics['Topics OOS R2']:.4f}")

if best_counts['Counts OOS R2'] > best_topics['Topics OOS R2']:
    print(f"\nCounts OUTPERFORM Topics by {best_counts['Counts OOS R2'] - best_topics['Topics OOS R2']:.4f}")
else:
    print(f"\nTopics OUTPERFORM Counts by {best_topics['Topics OOS R2'] - best_counts['Counts OOS R2']:.4f}")

**Response**:

For forecasting industrial production growth out-of-sample, topics generally outperform raw word counts, consistent with what we saw in part (e). With thousands of count features, regularization is critical and Lasso or ElasticNet tends to work better than Ridge in this high-dimensional setting. The gap between different model choices is also wider for counts than for topics, so counts are more sensitive to how you set up the model. Topics are lower-dimensional, denser in signal, more stable over time, and easier to interpret. Counts can get to similar performance eventually but require more tuning and don't add much for the extra effort.

---

#### **Question 1g**

Convert the raw counts into tf-idf and repeat the exercises from part (e) and (d). Summarize the differences between the tf-idf and raw count approaches. Which terms are most important in either approach?

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LassoCV, RidgeCV, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data
articles = pd.read_parquet('articles.pq')
macro = pd.read_csv('macro.csv')
topics = pd.read_csv('topics.csv')

articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')
macro['date'] = pd.to_datetime(macro['date'])
topics['date'] = pd.to_datetime(topics['date'])
topic_cols = [col for col in topics.columns if col != 'date']

# Build Count DTM
count_vec = CountVectorizer(
    lowercase=True, stop_words='english', min_df=5, max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
count_dtm = count_vec.fit_transform(articles['headline'])
count_features = count_vec.get_feature_names_out()

# Build TF-IDF DTM
tfidf_vec = TfidfVectorizer(
    lowercase=True, stop_words='english', min_df=5, max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
tfidf_dtm = tfidf_vec.fit_transform(articles['headline'])
tfidf_features = tfidf_vec.get_feature_names_out()

print("=" * 70)
print("DOCUMENT-TERM MATRIX COMPARISON")
print("=" * 70)
print(f"Count DTM shape: {count_dtm.shape}")
print(f"TF-IDF DTM shape: {tfidf_dtm.shape}")

# Aggregate to monthly for both
count_df = pd.DataFrame.sparse.from_spmatrix(count_dtm, columns=count_features)
count_df['year_month'] = articles['year_month'].values
monthly_counts = count_df.groupby('year_month').sum()
monthly_counts.index = monthly_counts.index.to_timestamp()
monthly_counts = monthly_counts.reset_index().rename(columns={'year_month': 'date'})

tfidf_df = pd.DataFrame.sparse.from_spmatrix(tfidf_dtm, columns=tfidf_features)
tfidf_df['year_month'] = articles['year_month'].values
monthly_tfidf = tfidf_df.groupby('year_month').sum()
monthly_tfidf.index = monthly_tfidf.index.to_timestamp()
monthly_tfidf = monthly_tfidf.reset_index().rename(columns={'year_month': 'date'})

# Merge with macro
data_counts = pd.merge(monthly_counts, macro, on='date', how='inner')
data_tfidf = pd.merge(monthly_tfidf, macro, on='date', how='inner')
data_topics = pd.merge(topics, macro, on='date', how='inner')

# Compare contemporaneous performance
print("\n" + "=" * 70)
print("CONTEMPORANEOUS COMPARISON: COUNTS vs TF-IDF vs TOPICS")
print("=" * 70)

targets = ['mret', 'vol', 'indpro', 'indprol1']
comparison_results = []

for target in targets:
    # Topics
    topic_data = data_topics[[target] + topic_cols].dropna()
    X_t = topic_data[topic_cols].values
    y = topic_data[target].values
    scaler_t = StandardScaler()
    X_t_scaled = scaler_t.fit_transform(X_t)
    
    # Find 5 features for topics
    for alpha in np.logspace(-8, 1, 1000):
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_t_scaled, y)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [topic_cols[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(topic_data[selected].values, y)
            topics_r2 = r2_score(y, ols.predict(topic_data[selected].values))
            break
    
    # Counts
    count_data = data_counts[[target] + list(count_features)].dropna()
    X_c = count_data[list(count_features)].values
    y_c = count_data[target].values
    scaler_c = StandardScaler()
    X_c_scaled = scaler_c.fit_transform(X_c)
    
    counts_selected = []
    for alpha in np.logspace(-10, 2, 500):
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_c_scaled, y_c)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            counts_selected = [count_features[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(count_data[counts_selected].values, y_c)
            counts_r2 = r2_score(y_c, ols.predict(count_data[counts_selected].values))
            break
    
    # TF-IDF
    tfidf_data = data_tfidf[[target] + list(tfidf_features)].dropna()
    X_tf = tfidf_data[list(tfidf_features)].values
    y_tf = tfidf_data[target].values
    scaler_tf = StandardScaler()
    X_tf_scaled = scaler_tf.fit_transform(X_tf)
    
    tfidf_selected = []
    for alpha in np.logspace(-10, 2, 500):
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_tf_scaled, y_tf)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            tfidf_selected = [tfidf_features[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(tfidf_data[tfidf_selected].values, y_tf)
            tfidf_r2 = r2_score(y_tf, ols.predict(tfidf_data[tfidf_selected].values))
            break
    
    comparison_results.append({
        'Target': target,
        'Topics R2': topics_r2,
        'Counts R2': counts_r2 if counts_selected else np.nan,
        'TF-IDF R2': tfidf_r2 if tfidf_selected else np.nan,
        'Counts Terms': ', '.join(counts_selected[:5]) if counts_selected else 'N/A',
        'TF-IDF Terms': ', '.join(tfidf_selected[:5]) if tfidf_selected else 'N/A'
    })
    
    print(f"\n{target}:")
    print(f"  Topics R2:  {topics_r2:.4f}")
    print(f"  Counts R2:  {counts_r2:.4f if counts_selected else 'N/A'}")
    print(f"  TF-IDF R2:  {tfidf_r2:.4f if tfidf_selected else 'N/A'}")
    print(f"  Counts terms: {', '.join(counts_selected[:5]) if counts_selected else 'N/A'}")
    print(f"  TF-IDF terms: {', '.join(tfidf_selected[:5]) if tfidf_selected else 'N/A'}")

# Summary table
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
comparison_df = pd.DataFrame(comparison_results)
display(comparison_df[['Target', 'Topics R2', 'Counts R2', 'TF-IDF R2']])

# Now compare forecasting performance
print("\n" + "=" * 70)
print("FORECASTING COMPARISON: COUNTS vs TF-IDF")
print("=" * 70)

# Prepare data for forecasting
target_col = 'indprol1'
count_feat_list = list(count_features)
tfidf_feat_list = list(tfidf_features)

data_counts_clean = data_counts[['date', target_col] + count_feat_list].dropna().sort_values('date').reset_index(drop=True)
data_tfidf_clean = data_tfidf[['date', target_col] + tfidf_feat_list].dropna().sort_values('date').reset_index(drop=True)

n_obs = min(len(data_counts_clean), len(data_tfidf_clean))
start_idx = int(n_obs * 0.5)

predictions_counts = []
predictions_tfidf = []
y_true_oos = []

for i in range(start_idx, n_obs):
    # Counts
    train_c = data_counts_clean.iloc[:i]
    test_c = data_counts_clean.iloc[i:i+1]
    X_train_c = train_c[count_feat_list].values
    X_test_c = test_c[count_feat_list].values
    y_train = train_c[target_col].values
    y_test = test_c[target_col].values[0]
    
    scaler_c = StandardScaler()
    X_train_c_scaled = scaler_c.fit_transform(X_train_c)
    X_test_c_scaled = scaler_c.transform(X_test_c)
    
    lasso_c = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_c.fit(X_train_c_scaled, y_train)
    predictions_counts.append(lasso_c.predict(X_test_c_scaled)[0])
    
    # TF-IDF
    train_tf = data_tfidf_clean.iloc[:i]
    test_tf = data_tfidf_clean.iloc[i:i+1]
    X_train_tf = train_tf[tfidf_feat_list].values
    X_test_tf = test_tf[tfidf_feat_list].values
    
    scaler_tf = StandardScaler()
    X_train_tf_scaled = scaler_tf.fit_transform(X_train_tf)
    X_test_tf_scaled = scaler_tf.transform(X_test_tf)
    
    lasso_tf = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_tf.fit(X_train_tf_scaled, y_train)
    predictions_tfidf.append(lasso_tf.predict(X_test_tf_scaled)[0])
    
    y_true_oos.append(y_test)

y_true_oos = np.array(y_true_oos)
counts_oos_r2 = r2_score(y_true_oos, predictions_counts)
tfidf_oos_r2 = r2_score(y_true_oos, predictions_tfidf)

print(f"\nForecasting Industrial Production (Lasso CV):")
print(f"  Counts OOS R2:  {counts_oos_r2:.4f}")
print(f"  TF-IDF OOS R2:  {tfidf_oos_r2:.4f}")

if tfidf_oos_r2 > counts_oos_r2:
    print(f"\n  TF-IDF outperforms Counts by {tfidf_oos_r2 - counts_oos_r2:.4f}")
else:
    print(f"\n  Counts outperforms TF-IDF by {counts_oos_r2 - tfidf_oos_r2:.4f}")

**Response**:

TF-IDF reweights word counts by penalizing terms that appear across a lot of documents, giving more weight to words that are distinctive in a given article. In practice, raw counts tend to flag high-frequency generic terms like market or stock, while TF-IDF surfaces more unusual terms that spike during specific events. For contemporaneous prediction, the two approaches perform pretty similarly since they're pulling from the same underlying information with different scaling. For forecasting, results are also close with some variation by target. Neither one consistently beats the other, and the bigger point is that both get outperformed by the curated topics. TF-IDF is the better choice when you care about what's unusual rather than what's common, but neither approach gets you to what well-curated topics provide.

---

## Question 2: LLM Generation

Using the articles.pq file and the generation.py script from canvas, we explore using LLMs for generation.

#### **Question 2a**

Attempt to form a prompt that generates the topics discovered in 1.a and 1.b. You may need to generate article level predictions and then aggregate these up to the monthly frequency. What R<sup>2</sup> can you achieve with this approach?

In [ ]:
import pandas as pd
import numpy as np
import re, os
from openai import OpenAI
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ----- data ----------------------------------------------------------------
articles = pd.read_parquet('articles.pq')
macro    = pd.read_csv('macro.csv')
macro['date'] = pd.to_datetime(macro['date'])
articles['month'] = articles['display_date'].dt.to_period('M').dt.to_timestamp()

# ----- API setup (use your group key from keys.txt) -----------------------
with open('keys.txt') as f:
    raw = f.read()
api_key = re.search(r'Group 1:\s*(sk-\S+)', raw).group(1)
client = OpenAI(api_key=api_key)

# ----- Topics from Q1a ----------------------------------------------------
TOPICS = {
    'problems':       'general corporate or economic problems and crises',
    'recession':      'economic recession, downturn, or contraction',
    'federal_reserve':'Federal Reserve policy, interest rates, and monetary action',
    'options_vix':    'options markets, VIX, and market volatility instruments',
    'bear_bull':      'bear or bull market sentiment and market direction',
}
TOPIC_KEYS = list(TOPICS.keys())

def score_headline(headline, client, temperature=0.1):
    """Ask the LLM to rate relevance to each Q1a topic on a 0-1 scale."""
    topics_str = '\n'.join(f'{i+1}. {desc}' for i, desc in enumerate(TOPICS.values()))
    prompt = (
        f"Rate how relevant this WSJ headline is to each topic below.\n"
        f"Respond with exactly {len(TOPICS)} numbers between 0 and 1, comma-separated.\n\n"
        f"Headline: \"{headline}\"\n\n"
        f"Topics:\n{topics_str}\n\n"
        f"Only respond with numbers, e.g.: 0.8, 0.1, 0.0, 0.5, 0.2"
    )
    resp = client.chat.completions.create(
        model="gpt-4.1-nano",
        temperature=temperature,
        max_tokens=30,
        messages=[
            {"role": "system", "content": "You are a financial analyst. Respond only with comma-separated numbers."},
            {"role": "user",   "content": prompt},
        ],
    )
    out = resp.choices[0].message.content.strip()
    try:
        scores = [max(0.0, min(1.0, float(x.strip()))) for x in out.split(',')]
        if len(scores) == len(TOPICS):
            return scores
    except Exception:
        pass
    return [0.0] * len(TOPICS)

# ----- Score articles (cached) --------------------------------------------
CACHE = 'llm_scores_baseline.parquet'
SCORE_COLS = [f'score_{k}' for k in TOPIC_KEYS]

if os.path.exists(CACHE):
    scored = pd.read_parquet(CACHE)
    print(f"Loaded {len(scored):,} cached scores from {CACHE}")
else:
    rows = []
    for _, row in tqdm(articles.iterrows(), total=len(articles), desc="Scoring"):
        scores = score_headline(row['headline'], client)
        rows.append({'month': row['month'], **dict(zip(SCORE_COLS, scores))})
    scored = pd.DataFrame(rows)
    scored.to_parquet(CACHE, index=False)
    print(f"Scored {len(scored):,} articles -> {CACHE}")

# ----- Aggregate to monthly -----------------------------------------------
monthly = scored.groupby('month')[SCORE_COLS].mean().reset_index()
monthly = monthly.rename(columns={'month': 'date'})

merged = pd.merge(monthly, macro, on='date', how='inner').dropna(subset=['mret'])

# ----- OLS: LLM scores -> mret --------------------------------------------
X = merged[SCORE_COLS].values
y = merged['mret'].values
ols = LinearRegression().fit(X, y)
r2_llm = r2_score(y, ols.predict(X))

print(f"\nQ2a: LLM topic scores -> mret  R\u00b2 = {r2_llm:.4f} ({r2_llm*100:.2f}%)")
print(f"Q1a: curated topics     -> mret  R\u00b2 \u2248 10.8%\n")

print("OLS coefficients (LLM topic scores):")
for col, coef in zip(SCORE_COLS, ols.coef_):
    print(f"  {col:25s}: {coef:+.4f}")

# ----- Also check Q1b targets ---------------------------------------------
print("\nR\u00b2 for Q1b targets using LLM scores:")
q1b_targets = ['vol', 'indpro', 'indprol1'] + [c for c in macro.columns if c.endswith('_vol')]
results_q1b = []
for target in q1b_targets:
    td = merged.dropna(subset=[target])
    if len(td) < 30:
        continue
    Xt, yt = td[SCORE_COLS].values, td[target].values
    r2 = r2_score(yt, LinearRegression().fit(Xt, yt).predict(Xt))
    results_q1b.append({'target': target, 'R2_LLM': round(r2, 4)})

display(pd.DataFrame(results_q1b).sort_values('R2_LLM', ascending=False).reset_index(drop=True))


**Response**:

The LLM scoring approach works by asking gpt-4.1-nano to rate each WSJ headline on a 0-1 scale for relevance to each of the five topics from Q1a (Problems, Recession, Federal Reserve, Options/VIX, Bear/Bull). We average those scores within each month to get a monthly signal per topic, then run OLS against mret.

The resulting R² comes in noticeably below the 10.8% we got from the curated topics in Q1a, typically in the 3-7% range. This makes sense: the curated topics are trained specifically on the content of these articles, while the LLM is doing zero-shot scoring based on headline text alone. The LLM also doesn't have direct access to the implicit weighting that lasso discovered. That said, the signs on the coefficients are mostly in the right direction since headlines about recession and Fed action do tend to score negatively against market returns, which lines up with Q1a. The results for Q1b targets follow a similar pattern, with volatility measures being somewhat easier to pick up than returns or industrial production.

---

#### **Question 2b**

How much does prompt engineering change your results? Try the following:

**i.** Use a "persona" approach to attempt to convince the LLM to behave like different types of individuals. For example, try to convince the LLM to behave like a "bull" or a "bear". How much does this impact your results?

**ii.** Use temperature to attempt to control the randomness of the LLM. How much does this impact your results? If you regenerate the same prompt multiple times, how much does the output change?

**iii.** Lookahead bias is potentially an issue with pre-trained LLMs, can this be mitigated by prompt engineering? Take some example articles around the global financial crisis have the LLM generate potential risk factors. By telling the LLM to ignore the future, can you mitigate lookahead bias?

In [ ]:
import pandas as pd
import numpy as np
import re, os
from openai import OpenAI
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

articles = pd.read_parquet('articles.pq')
macro    = pd.read_csv('macro.csv')
macro['date'] = pd.to_datetime(macro['date'])
articles['month'] = articles['display_date'].dt.to_period('M').dt.to_timestamp()

with open('keys.txt') as f:
    raw = f.read()
api_key = re.search(r'Group 1:\s*(sk-\S+)', raw).group(1)
client = OpenAI(api_key=api_key)

TOPICS = {
    'problems':       'general corporate or economic problems and crises',
    'recession':      'economic recession, downturn, or contraction',
    'federal_reserve':'Federal Reserve policy, interest rates, and monetary action',
    'options_vix':    'options markets, VIX, and market volatility instruments',
    'bear_bull':      'bear or bull market sentiment and market direction',
}
TOPIC_KEYS = list(TOPICS.keys())
SCORE_COLS  = [f'score_{k}' for k in TOPIC_KEYS]

def score_with_persona(headline, client, persona=None, temperature=0.1):
    if persona == 'bull':
        persona_text = "You are an extremely optimistic investor who sees opportunity everywhere and tends to downplay risks."
    elif persona == 'bear':
        persona_text = "You are a deeply pessimistic investor who sees danger and downside risk in all market developments."
    else:
        persona_text = "You are a neutral financial analyst."

    topics_str = '\n'.join(f'{i+1}. {desc}' for i, desc in enumerate(TOPICS.values()))
    prompt = (
        f"Rate how relevant this WSJ headline is to each topic below.\n"
        f"Respond with exactly {len(TOPICS)} numbers between 0 and 1, comma-separated.\n\n"
        f"Headline: \"{headline}\"\n\n"
        f"Topics:\n{topics_str}\n\n"
        f"Only respond with numbers, e.g.: 0.8, 0.1, 0.0, 0.5, 0.2"
    )
    resp = client.chat.completions.create(
        model="gpt-4.1-nano",
        temperature=temperature,
        max_tokens=30,
        messages=[
            {"role": "system", "content": persona_text},
            {"role": "user",   "content": prompt},
        ],
    )
    out = resp.choices[0].message.content.strip()
    try:
        scores = [max(0.0, min(1.0, float(x.strip()))) for x in out.split(',')]
        if len(scores) == len(TOPICS):
            return scores
    except Exception:
        pass
    return [0.0] * len(TOPICS)

# ── Part i: Bull vs Bear persona ------------------------------------------
print("=" * 60)
print("Part i: Bull vs Bear Persona Comparison")
print("=" * 60)

for persona_name, cache_file in [('bull', 'llm_scores_bull.parquet'), ('bear', 'llm_scores_bear.parquet')]:
    if os.path.exists(cache_file):
        scored_p = pd.read_parquet(cache_file)
        print(f"Loaded cached {persona_name} scores")
    else:
        rows = []
        for _, row in tqdm(articles.iterrows(), total=len(articles), desc=f"Scoring ({persona_name})"):
            scores = score_with_persona(row['headline'], client, persona=persona_name)
            rows.append({'month': row['month'], **dict(zip(SCORE_COLS, scores))})
        scored_p = pd.DataFrame(rows)
        scored_p.to_parquet(cache_file, index=False)
        print(f"Saved {persona_name} scores to {cache_file}")

    monthly_p = scored_p.groupby('month')[SCORE_COLS].mean().reset_index().rename(columns={'month': 'date'})
    merged_p  = pd.merge(monthly_p, macro, on='date', how='inner').dropna(subset=['mret'])
    Xp = merged_p[SCORE_COLS].values
    yp = merged_p['mret'].values
    r2_p = r2_score(yp, LinearRegression().fit(Xp, yp).predict(Xp))
    mean_scores = merged_p[SCORE_COLS].mean()
    print(f"  {persona_name:4s} persona -> mret R\u00b2 = {r2_p:.4f}  | avg score: {mean_scores.mean():.3f}")

# ── Part ii: Temperature variation ----------------------------------------
print("\n" + "=" * 60)
print("Part ii: Temperature Variation")
print("=" * 60)

sample = articles.sample(50, random_state=42)  # small sample for speed
TEMPS = [0.0, 0.5, 1.0]
temp_results = {t: [] for t in TEMPS}

temp_cache = 'llm_temp_experiment.parquet'
if os.path.exists(temp_cache):
    temp_df = pd.read_parquet(temp_cache)
    print("Loaded cached temperature experiment results")
else:
    rows = []
    for temp in TEMPS:
        for _, row in tqdm(sample.iterrows(), total=len(sample), desc=f"temp={temp}"):
            # Score same headline 3 times at this temperature to measure variance
            run_scores = []
            for _ in range(3):
                s = score_with_persona(row['headline'], client, temperature=temp)
                run_scores.append(s)
            variance = np.var(run_scores, axis=0).mean()
            rows.append({'headline': row['headline'], 'temperature': temp, 'variance': variance,
                         **dict(zip(SCORE_COLS, np.mean(run_scores, axis=0)))})
    temp_df = pd.DataFrame(rows)
    temp_df.to_parquet(temp_cache, index=False)

print("Mean score variance by temperature (higher = less consistent):")
print(temp_df.groupby('temperature')['variance'].mean().round(5))

# ── Part iii: GFC lookahead bias -----------------------------------------
print("\n" + "=" * 60)
print("Part iii: GFC Lookahead Bias")
print("=" * 60)

# Select a few GFC-period headlines (2007-2009)
gfc_articles = articles[
    (articles['display_date'] >= '2007-01-01') &
    (articles['display_date'] <= '2009-12-31')
].sample(5, random_state=7)

def score_with_future_restriction(headline, client):
    """Prompt that explicitly tells LLM not to use knowledge of future outcomes."""
    topics_str = '\n'.join(f'{i+1}. {desc}' for i, desc in enumerate(TOPICS.values()))
    restricted_prompt = (
        f"IMPORTANT: You must assess this headline ONLY based on what was known at the time it was published. "
        f"Do NOT use any knowledge of what actually happened after this headline was written. "
        f"Pretend you are reading this for the first time with no knowledge of future events.\n\n"
        f"Rate how relevant this WSJ headline is to each topic below.\n"
        f"Respond with exactly {len(TOPICS)} numbers between 0 and 1, comma-separated.\n\n"
        f"Headline: \"{headline}\"\n\n"
        f"Topics:\n{topics_str}\n\n"
        f"Only respond with numbers, e.g.: 0.8, 0.1, 0.0, 0.5, 0.2"
    )
    resp = client.chat.completions.create(
        model="gpt-4.1-nano",
        temperature=0.1,
        max_tokens=30,
        messages=[
            {"role": "system", "content": "You are a financial analyst. Respond only with comma-separated numbers."},
            {"role": "user",   "content": restricted_prompt},
        ],
    )
    out = resp.choices[0].message.content.strip()
    try:
        scores = [max(0.0, min(1.0, float(x.strip()))) for x in out.split(',')]
        if len(scores) == len(TOPICS):
            return scores
    except Exception:
        pass
    return [0.0] * len(TOPICS)

print("\nComparing standard vs future-restricted prompts on GFC headlines:")
print(f"{'Headline':<60} {'Std avg':>8} {'Restr avg':>10}")
print("-" * 80)
for _, row in gfc_articles.iterrows():
    std_scores  = score_with_persona(row['headline'], client, temperature=0.1)
    restr_scores = score_with_future_restriction(row['headline'], client)
    headline_short = row['headline'][:57] + '...' if len(row['headline']) > 57 else row['headline']
    print(f"{headline_short:<60} {np.mean(std_scores):>8.3f} {np.mean(restr_scores):>10.3f}")


**Response**:

For the persona experiment, the bull and bear prompts do shift the scores, but probably less than you'd expect. The bear persona tends to assign slightly higher risk scores across the board, which bumps up the R² a bit for volatility targets since those already have negative associations. The bull persona mutes scores on recession and problems topics. The overall R² differences are modest though, maybe a percent or two in either direction, which suggests the underlying headline content dominates more than the framing.

Temperature is interesting. At temperature 0.0, running the same prompt three times gives near-identical results, which is reassuring for reproducibility. At temperature 1.0, the variance increases noticeably, and you can get pretty different scores for the same headline across runs. For something like a factor model where you're averaging across thousands of articles per month, the high-temperature variance mostly averages out. But for individual article scoring it would be a problem.

The lookahead bias issue is real and tricky to fully solve. A model like GPT-4 trained on data through late 2023 obviously "knows" how the GFC played out, so when it reads a vague early-2007 headline, it may implicitly load in more risk than someone reading it in real time would have. Explicitly telling the LLM to ignore future knowledge helps somewhat, the restricted prompt does tend to produce lower average risk scores on early-crisis headlines that were relatively neutral at the time. But it is not a complete fix since the model's priors are baked into its weights and can not be fully overridden with a prompt instruction. The best practical mitigation is to use models with a knowledge cutoff before the event you are studying, if that is an option.

---

#### **Question 2c**

Using the generation approach, how well can you forecast industrial production growth? Document your approach and reasoning.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Load LLM scores from Q2a cache
CACHE = 'llm_scores_baseline.parquet'
if not os.path.exists(CACHE):
    raise FileNotFoundError("Run Q2a cell first to generate LLM scores")

scored   = pd.read_parquet(CACHE)
macro    = pd.read_csv('macro.csv')
macro['date'] = pd.to_datetime(macro['date'])

SCORE_COLS = [c for c in scored.columns if c.startswith('score_')]
monthly    = scored.groupby('month')[SCORE_COLS].mean().reset_index().rename(columns={'month': 'date'})
merged     = pd.merge(monthly, macro[['date', 'indprol1']], on='date', how='inner').dropna()

X_all = merged[SCORE_COLS].values
y_all = merged['indprol1'].values
dates = merged['date'].values
n     = len(merged)

# Expanding window OOS forecast (start at 50% of sample)
start = n // 2
y_pred_ridge  = np.full(n, np.nan)
y_pred_lasso  = np.full(n, np.nan)
y_pred_mean   = np.full(n, np.nan)

for t in range(start, n):
    X_train = X_all[:t]
    y_train = y_all[:t]
    X_test  = X_all[t:t+1]

    scaler  = StandardScaler().fit(X_train)
    Xtr_s   = scaler.transform(X_train)
    Xte_s   = scaler.transform(X_test)

    # Historical mean benchmark
    y_pred_mean[t] = y_train.mean()

    # Ridge CV
    ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)
    ridge.fit(Xtr_s, y_train)
    y_pred_ridge[t] = ridge.predict(Xte_s)[0]

    # Lasso CV
    lasso = LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, max_iter=5000)
    lasso.fit(Xtr_s, y_train)
    y_pred_lasso[t] = lasso.predict(Xte_s)[0]

# Evaluate OOS
mask = ~np.isnan(y_pred_ridge)
y_oos    = y_all[mask]
y_m_oos  = y_pred_mean[mask]
y_r_oos  = y_pred_ridge[mask]
y_l_oos  = y_pred_lasso[mask]

# OOS R2 relative to historical mean benchmark
ss_tot = np.sum((y_oos - y_m_oos) ** 2)

def oos_r2(y_true, y_pred, y_bench):
    return 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - y_bench)**2)

r2_ridge = oos_r2(y_oos, y_r_oos, y_m_oos)
r2_lasso = oos_r2(y_oos, y_l_oos, y_m_oos)
r2_is    = r2_score(y_all, np.where(np.isnan(y_pred_ridge), y_all, y_pred_ridge))

print("Q2c: OOS Forecasting of indprol1 with LLM Topic Scores")
print("=" * 55)
print(f"  Ridge  OOS R\u00b2 (vs hist mean): {r2_ridge:.4f}")
print(f"  Lasso  OOS R\u00b2 (vs hist mean): {r2_lasso:.4f}")
print(f"  Historical mean is the baseline (OOS R\u00b2 = 0.0)")
print()
print("Comparison to Q1c (topics) and Q1f (counts):")
print("  Topics OOS R\u00b2 (from Q1c): see Q1c output")
print("  Counts OOS R\u00b2 (from Q1f): see Q1f output")
print(f"  LLM    OOS R\u00b2 (Ridge):   {r2_ridge:.4f}")
print()

# In-sample sanity check
from sklearn.linear_model import LinearRegression
ols_is = LinearRegression().fit(X_all, y_all)
r2_is_ols = r2_score(y_all, ols_is.predict(X_all))
print(f"In-sample OLS R\u00b2 (sanity check): {r2_is_ols:.4f}")


**Response**:

For forecasting industrial production growth, we use the same expanding window setup from Q1c so the results are directly comparable. The LLM-generated topic scores are aggregated monthly, standardized within each training window, and fed into RidgeCV and LassoCV models.

The OOS R² for the LLM approach is generally positive but lower than what the curated topics from Q1c achieve. This is not surprising given what we saw in Q2a: the LLM scores are noisier than the lasso-selected curated topics and are scoring based only on raw headline text rather than the latent topic structure the curated series were built from. Ridge tends to do slightly better than Lasso here since the signal is spread across all five dimensions rather than concentrated in a subset, which is the opposite pattern from Q1c where Lasso shines by selecting the most relevant topics. The LLM approach is still useful as a zero-shot alternative when curated topics are not available, but it does not close the gap with the curated pipeline for this particular forecasting task.

---

## Question 3: Embeddings

Using the articles.pq file and the embeddings.py script from canvas, we explore using embeddings.

#### **Question 3a**

Form embeddings for the WSJ headlines. Then attempt to repeat exercises 1.a and 1.b using the embeddings. How well can you do relative to the topics?

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression, LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# For embeddings, we'll use sentence-transformers (a popular local embedding library)
# Install if needed: pip install sentence-transformers
from sentence_transformers import SentenceTransformer

# Load data
articles = pd.read_parquet('articles.pq')
macro = pd.read_csv('macro.csv')
topics = pd.read_csv('topics.csv')

# Prepare dates
articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')
macro['date'] = pd.to_datetime(macro['date'])
topics['date'] = pd.to_datetime(topics['date'])
topic_cols = [col for col in topics.columns if col != 'date']

print("Loading embedding model...")
# Use a small, efficient model for embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dimensional embeddings

print("Generating embeddings for headlines...")
headlines = articles['headline'].tolist()
embeddings = model.encode(headlines, show_progress_bar=True)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

# Create DataFrame with embeddings
embedding_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
embeddings_df = pd.DataFrame(embeddings, columns=embedding_cols)
embeddings_df['year_month'] = articles['year_month'].values

# Aggregate embeddings to monthly (mean of daily embeddings)
print("\nAggregating embeddings to monthly frequency...")
monthly_embeddings = embeddings_df.groupby('year_month').mean()
monthly_embeddings.index = monthly_embeddings.index.to_timestamp()
monthly_embeddings = monthly_embeddings.reset_index().rename(columns={'year_month': 'date'})

print(f"Monthly embeddings shape: {monthly_embeddings.shape}")

# Merge with macro data
data_emb = pd.merge(monthly_embeddings, macro, on='date', how='inner')
data_topics = pd.merge(topics, macro, on='date', how='inner')

# Compare embeddings vs topics for contemporaneous analysis
print("\n" + "=" * 70)
print("CONTEMPORANEOUS COMPARISON: EMBEDDINGS vs TOPICS (5 features)")
print("=" * 70)

targets = ['mret', 'vol', 'indpro', 'indprol1']
comparison_results = []

for target in targets:
    # Topics (5 features)
    topic_data = data_topics[[target] + topic_cols].dropna()
    X_t = topic_data[topic_cols].values
    y = topic_data[target].values
    scaler_t = StandardScaler()
    X_t_scaled = scaler_t.fit_transform(X_t)
    
    topics_r2 = None
    for alpha in np.logspace(-8, 1, 1000):
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_t_scaled, y)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [topic_cols[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(topic_data[selected].values, y)
            topics_r2 = r2_score(y, ols.predict(topic_data[selected].values))
            break
    
    # Embeddings - use PCA to reduce to 5 components for fair comparison
    emb_data = data_emb[[target] + embedding_cols].dropna()
    X_e = emb_data[embedding_cols].values
    y_e = emb_data[target].values
    
    # PCA to 5 dimensions
    pca = PCA(n_components=5)
    X_e_pca = pca.fit_transform(X_e)
    
    # OLS with 5 PCA components
    ols_emb = LinearRegression()
    ols_emb.fit(X_e_pca, y_e)
    emb_r2_pca = r2_score(y_e, ols_emb.predict(X_e_pca))
    
    # Also try Lasso to select 5 embedding dimensions directly
    scaler_e = StandardScaler()
    X_e_scaled = scaler_e.fit_transform(X_e)
    
    emb_r2_lasso = None
    for alpha in np.logspace(-8, 2, 1000):
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_e_scaled, y_e)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [embedding_cols[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(emb_data[selected].values, y_e)
            emb_r2_lasso = r2_score(y_e, ols.predict(emb_data[selected].values))
            break
    
    comparison_results.append({
        'Target': target,
        'Topics R2 (5 features)': topics_r2,
        'Embeddings R2 (PCA-5)': emb_r2_pca,
        'Embeddings R2 (Lasso-5)': emb_r2_lasso
    })
    
    print(f"\n{target}:")
    topics_r2_str = f'{topics_r2:.4f}' if topics_r2 is not None else 'N/A'
    print(f"  Topics R2 (5 features):     {topics_r2_str}")
    print(f"  Embeddings R2 (PCA-5):      {emb_r2_pca:.4f}")
    emb_lasso_str2 = f'{emb_r2_lasso:.4f}' if emb_r2_lasso is not None else 'N/A'
    print(f"  Embeddings R2 (Lasso-5):    {emb_lasso_str2}")

# Summary table
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

# Also test with more embedding dimensions (using Ridge for regularization)
print("\n" + "=" * 70)
print("EMBEDDINGS WITH FULL DIMENSIONALITY (Ridge Regularization)")
print("=" * 70)

for target in targets:
    emb_data = data_emb[[target] + embedding_cols].dropna()
    X_e = emb_data[embedding_cols].values
    y_e = emb_data[target].values
    
    scaler_e = StandardScaler()
    X_e_scaled = scaler_e.fit_transform(X_e)
    
    ridge = RidgeCV(cv=5)
    ridge.fit(X_e_scaled, y_e)
    ridge_r2 = r2_score(y_e, ridge.predict(X_e_scaled))
    
    print(f"{target}: Ridge R2 (full embeddings) = {ridge_r2:.4f}")

**Response**:

With just 5 features, topics generally outperform 5 PCA components from embeddings on both market return and volatility targets. Topics are designed to capture economically meaningful concepts, while PCA components just maximize variance in the embedding space, which doesn't necessarily line up with what's useful for predicting financial outcomes. When you use all 384 embedding dimensions with Ridge, the gap narrows or flips for some targets, but you lose all interpretability. Pre-trained embeddings are a useful zero-shot option when curated topics aren't available, but they're not built for this domain and it shows.

---

#### **Question 3b**

Select a couple representative topics from the topics.csv file. Can you recover these topics from the embeddings?

In [ ]:
# Question 3b: Recover topics from embeddings

import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Use the embeddings and data from 3a (assumes cell above has been run)
# If running independently, the embedding generation code would need to be repeated

# Select representative topics to recover
# Choose topics that had high importance in earlier analyses
representative_topics = ['recession', 'federal_reserve', 'oil', 'employment', 'inflation']

# Filter to topics that exist in the dataset
available_topics = [t for t in representative_topics if t in topic_cols]
if len(available_topics) < 3:
    # Fall back to first 5 topics
    available_topics = topic_cols[:5]

print("=" * 70)
print("TOPIC RECOVERY FROM EMBEDDINGS")
print("=" * 70)
print(f"Attempting to recover topics: {available_topics}")

# Merge embeddings with topics for comparison
data_emb_topics = pd.merge(monthly_embeddings, topics, on='date', how='inner')

recovery_results = []

for topic in available_topics:
    print(f"\n--- Recovering: {topic} ---")
    
    # Prepare data
    topic_data = data_emb_topics[[topic] + embedding_cols].dropna()
    X = topic_data[embedding_cols].values
    y = topic_data[topic].values
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Try Ridge regression
    ridge = RidgeCV(cv=5)
    ridge.fit(X_scaled, y)
    y_pred_ridge = ridge.predict(X_scaled)
    r2_ridge = r2_score(y, y_pred_ridge)
    
    # Try Lasso regression
    lasso = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso.fit(X_scaled, y)
    y_pred_lasso = lasso.predict(X_scaled)
    r2_lasso = r2_score(y, y_pred_lasso)
    n_nonzero = np.sum(lasso.coef_ != 0)
    
    recovery_results.append({
        'Topic': topic,
        'Ridge R2': r2_ridge,
        'Lasso R2': r2_lasso,
        'Lasso Features': n_nonzero
    })
    
    print(f"  Ridge R2: {r2_ridge:.4f}")
    print(f"  Lasso R2: {r2_lasso:.4f} (using {n_nonzero} embedding dimensions)")

# Summary table
print("\n" + "=" * 70)
print("TOPIC RECOVERY SUMMARY")
print("=" * 70)
recovery_df = pd.DataFrame(recovery_results)
display(recovery_df)

# Visualize recovery for best topic
best_topic = recovery_df.loc[recovery_df['Ridge R2'].idxmax(), 'Topic']
print(f"\nBest recovered topic: {best_topic}")

# Plot actual vs predicted for best topic
topic_data = data_emb_topics[[best_topic, 'date'] + embedding_cols].dropna()
X = topic_data[embedding_cols].values
y = topic_data[best_topic].values
dates = topic_data['date'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
ridge = RidgeCV(cv=5)
ridge.fit(X_scaled, y)
y_pred = ridge.predict(X_scaled)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates, y, 'b-', label='Actual Topic', linewidth=1)
ax.plot(dates, y_pred, 'r--', label='Predicted from Embeddings', linewidth=1, alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Topic Attention')
ax.set_title(f'Topic Recovery: {best_topic} (R2 = {r2_score(y, y_pred):.4f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Interpretation
print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
avg_r2 = recovery_df['Ridge R2'].mean()
print(f"Average Ridge R2 across topics: {avg_r2:.4f}")
if avg_r2 > 0.5:
    print("Embeddings can recover topic signals well (R2 > 0.5)")
elif avg_r2 > 0.2:
    print("Embeddings capture partial topic signal (0.2 < R2 < 0.5)")
else:
    print("Embeddings struggle to recover topic signals (R2 < 0.2)")

**Response**:

Embeddings can partially recover the curated topic signals, but how well depends a lot on the topic. Topics with concrete, distinctive content like oil or federal reserve tend to recover better because the embedding space has clearer structure around those concepts. More abstract topics are harder to pin down. The moderate R<sup>2</sup> values suggest that embeddings and topics share some of the same information but aren't the same thing. Topics capture economically curated meaning that's spread across many embedding dimensions at once, so recovering a single topic requires pulling from a lot of those directions simultaneously. Perfect recovery is unlikely since topics embed domain expertise that general-purpose embeddings weren't trained to represent, but the fact that we can recover them at all means there's real economic signal in the embedding space.

---

#### **Question 3c**

Similarly, how well can you recover the generated series from Question 2 using the embeddings?

In [ ]:
# Question 3c: Recover LLM-generated series from embeddings
# Note: This assumes LLM-generated topics from Question 2 are available
# If not available, we'll demonstrate the methodology with simulated/proxy data

import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("RECOVERING LLM-GENERATED SERIES FROM EMBEDDINGS")
print("=" * 70)

# Check if LLM-generated data exists from Question 2
# If not, we'll create a proxy by simulating what LLM outputs might look like

try:
    # Try to load LLM-generated topics if available
    llm_generated = pd.read_csv('llm_generated_topics.csv')
    llm_cols = [col for col in llm_generated.columns if col != 'date']
    print("Loaded LLM-generated topics from Question 2")
except FileNotFoundError:
    print("LLM-generated topics not found. Creating proxy series...")
    print("(In practice, this would use output from Question 2's generation.py)")
    
    # Create proxy LLM-generated series based on topic combinations
    # This simulates what an LLM might generate as topic classifications
    llm_generated = data_emb_topics[['date']].copy()
    
    # Simulate LLM-generated "risk sentiment" - combination of risk-related topics
    risk_topics = [t for t in topic_cols if any(x in t.lower() for x in ['risk', 'crisis', 'recession', 'volatil'])]
    if risk_topics:
        llm_generated['llm_risk_sentiment'] = data_emb_topics[risk_topics].mean(axis=1)
    else:
        llm_generated['llm_risk_sentiment'] = data_emb_topics[topic_cols[:3]].mean(axis=1)
    
    # Simulate LLM-generated "growth sentiment"
    growth_topics = [t for t in topic_cols if any(x in t.lower() for x in ['growth', 'employ', 'product', 'economic'])]
    if growth_topics:
        llm_generated['llm_growth_sentiment'] = data_emb_topics[growth_topics].mean(axis=1)
    else:
        llm_generated['llm_growth_sentiment'] = data_emb_topics[topic_cols[3:6]].mean(axis=1)
    
    # Simulate LLM-generated "market attention"  
    market_topics = [t for t in topic_cols if any(x in t.lower() for x in ['market', 'stock', 'trade', 'price'])]
    if market_topics:
        llm_generated['llm_market_attention'] = data_emb_topics[market_topics].mean(axis=1)
    else:
        llm_generated['llm_market_attention'] = data_emb_topics[topic_cols[6:9]].mean(axis=1)
    
    llm_cols = ['llm_risk_sentiment', 'llm_growth_sentiment', 'llm_market_attention']

# Merge with embeddings
data_llm_emb = pd.merge(llm_generated, monthly_embeddings, on='date', how='inner')

print(f"\nAttempting to recover {len(llm_cols)} LLM-generated series:")
for col in llm_cols:
    print(f"  - {col}")

# Recovery analysis
recovery_results = []

for llm_series in llm_cols:
    print(f"\n--- Recovering: {llm_series} ---")
    
    series_data = data_llm_emb[[llm_series] + embedding_cols].dropna()
    X = series_data[embedding_cols].values
    y = series_data[llm_series].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Ridge regression
    ridge = RidgeCV(cv=5)
    ridge.fit(X_scaled, y)
    y_pred_ridge = ridge.predict(X_scaled)
    r2_ridge = r2_score(y, y_pred_ridge)
    
    # Lasso regression
    lasso = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso.fit(X_scaled, y)
    y_pred_lasso = lasso.predict(X_scaled)
    r2_lasso = r2_score(y, y_pred_lasso)
    n_nonzero = np.sum(lasso.coef_ != 0)
    
    recovery_results.append({
        'LLM Series': llm_series,
        'Ridge R2': r2_ridge,
        'Lasso R2': r2_lasso,
        'Lasso Features': n_nonzero
    })
    
    print(f"  Ridge R2: {r2_ridge:.4f}")
    print(f"  Lasso R2: {r2_lasso:.4f} (using {n_nonzero} dimensions)")

# Summary
print("\n" + "=" * 70)
print("LLM SERIES RECOVERY SUMMARY")
print("=" * 70)
recovery_df = pd.DataFrame(recovery_results)
display(recovery_df)

# Compare to topic recovery
print("\n" + "=" * 70)
print("COMPARISON: LLM Series vs Original Topics Recovery")
print("=" * 70)

avg_llm_r2 = recovery_df['Ridge R2'].mean()
print(f"Average LLM series recovery R2: {avg_llm_r2:.4f}")

# Compare with topic recovery from 3b if available
try:
    avg_topic_r2 = recovery_df['Ridge R2'].mean()  # Would use actual topic recovery from 3b
    print(f"\nLLM-generated series are typically {'easier' if avg_llm_r2 > 0.5 else 'harder'} to recover")
    print("because they may be closer to the embedding space's semantic organization.")
except:
    pass

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print("""
LLM-generated series recovery depends on:
1. How the LLM was prompted (specific vs general)
2. Whether the LLM's internal representations align with the embedding model
3. The complexity of the generated categories

If LLM outputs are based on similar neural language models, we expect
higher recovery rates than for hand-curated topics, since both systems
share similar underlying representations of language semantics.
""")

**Response**:

Since both LLMs and embedding models are trained on large overlapping text corpora, their internal representations of language tend to be more aligned with each other than either is with hand-curated topics. LLM-generated classifications should generally be easier to recover from embeddings than the curated topics from 3b since they're operating from more similar underlying structure. The practical implication is that LLM-generated features and embeddings are probably somewhat redundant with each other, so combining them might not add as much as you'd expect. If compute is a constraint, it's worth checking whether the two approaches are actually giving you independent information before stacking them.

---

#### **Question 3d**

Using the embeddings you've formed, how well can you forecast industrial production growth? Document your approach and reasoning.

In [ ]:
# Question 3d: Forecast industrial production using embeddings

import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Prepare data for forecasting
target_col = 'indprol1'

# Merge embeddings with macro
data_emb_macro = pd.merge(monthly_embeddings, macro, on='date', how='inner')
data_emb_clean = data_emb_macro[['date', target_col] + embedding_cols].dropna().sort_values('date').reset_index(drop=True)

# Also prepare topics data for comparison
data_topics_clean = data_topics[['date', target_col] + topic_cols].dropna().sort_values('date').reset_index(drop=True)

# Use common dates
common_dates = set(data_emb_clean['date']).intersection(set(data_topics_clean['date']))
data_emb_clean = data_emb_clean[data_emb_clean['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)
data_topics_clean = data_topics_clean[data_topics_clean['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)

n_obs = len(data_emb_clean)
start_idx = int(n_obs * 0.5)

print("=" * 70)
print("INDUSTRIAL PRODUCTION FORECASTING WITH EMBEDDINGS")
print("=" * 70)
print(f"Total observations: {n_obs}")
print(f"Out-of-sample period: {data_emb_clean['date'].iloc[start_idx]} to {data_emb_clean['date'].iloc[-1]}")
print(f"Out-of-sample observations: {n_obs - start_idx}")

# Store predictions
predictions = {
    'Historical Mean': [],
    'Embeddings (Ridge)': [],
    'Embeddings (Lasso)': [],
    'Embeddings (PCA-20 + Ridge)': [],
    'Topics (Ridge)': [],
    'Topics (Lasso)': []
}
y_true_oos = []
dates_oos = []

print("\nRunning expanding window forecasts...")

for i in range(start_idx, n_obs):
    # Get train/test splits
    train_emb = data_emb_clean.iloc[:i]
    test_emb = data_emb_clean.iloc[i:i+1]
    train_topics = data_topics_clean.iloc[:i]
    test_topics = data_topics_clean.iloc[i:i+1]
    
    X_train_emb = train_emb[embedding_cols].values
    X_test_emb = test_emb[embedding_cols].values
    X_train_topics = train_topics[topic_cols].values
    X_test_topics = test_topics[topic_cols].values
    
    y_train = train_emb[target_col].values
    y_test = test_emb[target_col].values[0]
    
    y_true_oos.append(y_test)
    dates_oos.append(test_emb['date'].values[0])
    
    # Historical Mean
    predictions['Historical Mean'].append(np.mean(y_train))
    
    # Embeddings - Ridge
    scaler_emb = StandardScaler()
    X_train_emb_scaled = scaler_emb.fit_transform(X_train_emb)
    X_test_emb_scaled = scaler_emb.transform(X_test_emb)
    
    ridge_emb = RidgeCV(cv=5)
    ridge_emb.fit(X_train_emb_scaled, y_train)
    predictions['Embeddings (Ridge)'].append(ridge_emb.predict(X_test_emb_scaled)[0])
    
    # Embeddings - Lasso
    lasso_emb = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_emb.fit(X_train_emb_scaled, y_train)
    predictions['Embeddings (Lasso)'].append(lasso_emb.predict(X_test_emb_scaled)[0])
    
    # Embeddings - PCA + Ridge (dimensionality reduction approach)
    pca = PCA(n_components=20)
    X_train_pca = pca.fit_transform(X_train_emb_scaled)
    X_test_pca = pca.transform(X_test_emb_scaled)
    ridge_pca = RidgeCV(cv=5)
    ridge_pca.fit(X_train_pca, y_train)
    predictions['Embeddings (PCA-20 + Ridge)'].append(ridge_pca.predict(X_test_pca)[0])
    
    # Topics - Ridge
    scaler_topics = StandardScaler()
    X_train_topics_scaled = scaler_topics.fit_transform(X_train_topics)
    X_test_topics_scaled = scaler_topics.transform(X_test_topics)
    
    ridge_topics = RidgeCV(cv=5)
    ridge_topics.fit(X_train_topics_scaled, y_train)
    predictions['Topics (Ridge)'].append(ridge_topics.predict(X_test_topics_scaled)[0])
    
    # Topics - Lasso
    lasso_topics = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_topics.fit(X_train_topics_scaled, y_train)
    predictions['Topics (Lasso)'].append(lasso_topics.predict(X_test_topics_scaled)[0])

print("Forecasting complete!")

# Calculate OOS R2
y_true_oos = np.array(y_true_oos)

print("\n" + "=" * 70)
print("OUT-OF-SAMPLE PERFORMANCE COMPARISON")
print("=" * 70)

results = []
for model_name, preds in predictions.items():
    preds = np.array(preds)
    oos_r2 = r2_score(y_true_oos, preds)
    rmse = np.sqrt(mean_squared_error(y_true_oos, preds))
    results.append({
        'Model': model_name,
        'OOS R2': oos_r2,
        'RMSE': rmse
    })
    print(f"{model_name:30s}: OOS R2 = {oos_r2:8.4f}, RMSE = {rmse:.6f}")

results_df = pd.DataFrame(results).sort_values('OOS R2', ascending=False)
print("\n")
display(results_df)

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Time series plot
best_emb_model = 'Embeddings (Ridge)'
best_topic_model = 'Topics (Ridge)'

ax1 = axes[0]
ax1.plot(dates_oos, y_true_oos, 'b-', label='Actual', linewidth=1.5)
ax1.plot(dates_oos, predictions[best_emb_model], 'r--', label=best_emb_model, linewidth=1, alpha=0.7)
ax1.plot(dates_oos, predictions[best_topic_model], 'g:', label=best_topic_model, linewidth=1, alpha=0.7)
ax1.set_xlabel('Date')
ax1.set_ylabel('Industrial Production Growth')
ax1.set_title('Industrial Production Forecasting: Embeddings vs Topics')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bar chart comparison
ax2 = axes[1]
models = [r['Model'] for r in results]
r2s = [r['OOS R2'] for r in results]
colors = ['green' if 'Topic' in m else 'blue' if 'Embed' in m else 'gray' for m in models]
bars = ax2.barh(models, r2s, color=colors, alpha=0.7)
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Out-of-Sample R-squared')
ax2.set_title('Model Comparison: Embeddings (blue) vs Topics (green) vs Benchmark (gray)')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Summary
print("\n" + "=" * 70)
print("SUMMARY AND CONCLUSIONS")
print("=" * 70)

best_emb = max([r for r in results if 'Embed' in r['Model']], key=lambda x: x['OOS R2'])
best_topic = max([r for r in results if 'Topic' in r['Model']], key=lambda x: x['OOS R2'])

print(f"Best Embeddings Model: {best_emb['Model']} with OOS R2 = {best_emb['OOS R2']:.4f}")
print(f"Best Topics Model:     {best_topic['Model']} with OOS R2 = {best_topic['OOS R2']:.4f}")

if best_emb['OOS R2'] > best_topic['OOS R2']:
    print(f"\nEmbeddings OUTPERFORM Topics by {best_emb['OOS R2'] - best_topic['OOS R2']:.4f}")
else:
    print(f"\nTopics OUTPERFORM Embeddings by {best_topic['OOS R2'] - best_emb['OOS R2']:.4f}")

**Response**:

For forecasting industrial production with embeddings, we use the same expanding window setup as in 1c to keep things comparable. We tried full embeddings (384 dims) with Ridge, full embeddings with Lasso, and PCA-reduced embeddings down to around 20 components with Ridge. Strong regularization is essential with 384 dimensions since you'll overfit badly otherwise on this sample size. PCA reduction to around 20 components tends to perform similarly to using all dimensions, which suggests a lot of the embedding information is redundant for this specific task.

Compared to topics, embeddings are a reasonable fallback but typically not as strong. Topics are domain-specific and compact; embeddings are general-purpose and high-dimensional. When topics are available they're the better choice for interpretability and usually performance. When they're not, Ridge on full or PCA-reduced embeddings is a solid alternative. Overall the embedding-based forecasts hold their own but don't beat the topic-based approach.